# Hybrid LSTM Boyacá–Guayaquil: reproducible educational experiment

This notebook reproduces five-class and three-class congestion experiments with chronological partitions, class-sensitive metrics, baseline comparisons, and exportable artifacts.

**Important:** the dataset is synthetic, didactic, and non-official. Running this notebook generates metrics and models from the current runtime; no results are embedded or fabricated in the notebook.


## 1. Runtime and repository setup

Run in Google Colab with a GPU runtime. The setup cell clones the public repository when it is not already available and installs the declared dependencies.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/TheCat0/Modelo-h-brido-LSTM-para-predicci-n-de-congesti-n-urbana.git"
REPO_DIR = Path("/content/Modelo-h-brido-LSTM-para-predicci-n-de-congesti-n-urbana")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run(["python", "-m", "pip", "install", "-r", "requirements.txt"], check=True)
print("Repository:", REPO_DIR)
print("Working directory:", Path.cwd())


## 2. Imports, reproducibility controls, and environment audit


In [ ]:
import json
import platform
import random
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from tensorflow.keras import Model, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Concatenate, Dense, Dropout, LSTM

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

for directory in ("outputs", "models", "artifacts"):
    Path(directory).mkdir(parents=True, exist_ok=True)

environment = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "tensorflow": tf.__version__,
    "gpu_devices": [device.name for device in tf.config.list_physical_devices("GPU")],
    "seed": SEED,
}
Path("outputs/environment_summary.json").write_text(
    json.dumps(environment, indent=2, ensure_ascii=False), encoding="utf-8"
)
environment


## 3. Load and audit the synthetic Boyacá–Guayaquil dataset


In [ ]:
DATASET_PATH = Path("data/traffic_observations_boyaca_guayaquil_2026.csv")
REQUIRED_COLUMNS = [
    "timestamp", "site_id", "flow_total", "motos", "buses", "camiones",
    "velocidad_prom", "tiempo_recorrido", "cola", "incidente", "lluvia",
    "tipo_via", "carriles", "zona", "hora_pico", "feriado", "congestion",
]

df = pd.read_csv(DATASET_PATH)
missing = sorted(set(REQUIRED_COLUMNS) - set(df.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="raise")
df = df.sort_values(["site_id", "timestamp"]).reset_index(drop=True)

intervals = df.groupby("site_id")["timestamp"].diff().dropna()
if not intervals.eq(pd.Timedelta(minutes=15)).all():
    raise ValueError("The dataset is not consecutive at 15-minute intervals within each site.")

print("Rows:", len(df))
print("Sites:", df["site_id"].unique().tolist())
print("Range:", df["timestamp"].min(), "to", df["timestamp"].max())
display(df.head())
display(df["congestion"].value_counts().sort_index().rename("count").to_frame())


## 4. Temporal preparation helpers

The workflow splits each site chronologically before window creation and removes `sequence_length - 1` rows at partition boundaries. This prevents windows from different partitions from sharing timesteps.


In [ ]:
SEQUENCE_LENGTH = 8
TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20
SEQ_COLUMNS = [
    "flow_total", "motos", "buses", "camiones", "velocidad_prom",
    "tiempo_recorrido", "cola", "incidente", "lluvia",
]
NUMERIC_CONTEXT = ["carriles", "hora_pico", "feriado"]
CATEGORICAL_CONTEXT = ["tipo_via", "zona"]
CONTEXT_COLUMNS = NUMERIC_CONTEXT + CATEGORICAL_CONTEXT


def chronological_partitions(frame):
    gap = SEQUENCE_LENGTH - 1
    partitions = {"train": [], "validation": [], "test": []}
    ranges = {}
    for site_id, group in frame.groupby("site_id", sort=False):
        group = group.sort_values("timestamp").reset_index(drop=True)
        usable = len(group) - 2 * gap
        test_rows = max(SEQUENCE_LENGTH + 1, round(usable * TEST_SIZE))
        validation_rows = max(SEQUENCE_LENGTH + 1, round((usable - test_rows) * VALIDATION_SIZE))
        train_rows = usable - validation_rows - test_rows
        if min(train_rows, validation_rows, test_rows) <= SEQUENCE_LENGTH:
            raise ValueError(f"Insufficient rows for temporal partitions at site {site_id}.")
        validation_start = train_rows + gap
        test_start = validation_start + validation_rows + gap
        split = {
            "train": group.iloc[:train_rows].copy(),
            "validation": group.iloc[validation_start:validation_start + validation_rows].copy(),
            "test": group.iloc[test_start:test_start + test_rows].copy(),
        }
        for name, part in split.items():
            partitions[name].append(part)
        ranges[str(site_id)] = {
            name: [part["timestamp"].iloc[0].isoformat(), part["timestamp"].iloc[-1].isoformat()]
            for name, part in split.items()
        }
    return {name: pd.concat(parts, ignore_index=True) for name, parts in partitions.items()}, ranges


def create_windows(frame, target_column):
    sequence, context, target = [], [], []
    for _, group in frame.groupby("site_id", sort=False):
        group = group.sort_values("timestamp").reset_index(drop=True)
        seq_values = group[SEQ_COLUMNS].to_numpy(dtype=np.float32)
        ctx_values = group[CONTEXT_COLUMNS].to_numpy(dtype=object)
        target_values = group[target_column].to_numpy(dtype=np.int32)
        for index in range(SEQUENCE_LENGTH, len(group)):
            sequence.append(seq_values[index-SEQUENCE_LENGTH:index])
            context.append(ctx_values[index])
            target.append(target_values[index])
    return np.asarray(sequence, dtype=np.float32), np.asarray(context, dtype=object), np.asarray(target)


def dense_float32(values):
    if hasattr(values, "toarray"):
        values = values.toarray()
    return values.astype(np.float32)


## 5. Model and experiment functions

Each scenario fits preprocessing only on training rows. Baselines use the same flattened temporal and contextual inputs as the hybrid model. Metrics include accuracy, balanced accuracy, macro F1, and weighted F1.


In [ ]:
def build_hybrid_model(context_features, classes):
    sequence_input = Input((SEQUENCE_LENGTH, len(SEQ_COLUMNS)), name="seq_input")
    x = LSTM(64, return_sequences=True)(sequence_input)
    x = Dropout(0.20)(x)
    x = LSTM(32)(x)

    context_input = Input((context_features,), name="ctx_input")
    c = Dense(16, activation="relu")(context_input)
    c = Dropout(0.10)(c)

    merged = Concatenate()([x, c])
    merged = Dense(16, activation="relu")(merged)
    output = Dense(classes, activation="softmax")(merged)
    model = Model([sequence_input, context_input], output)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def metric_row(scenario, model_name, truth, prediction):
    return {
        "scenario": scenario,
        "model": model_name,
        "accuracy": accuracy_score(truth, prediction),
        "balanced_accuracy": balanced_accuracy_score(truth, prediction),
        "macro_f1": f1_score(truth, prediction, average="macro", zero_division=0),
        "weighted_f1": f1_score(truth, prediction, average="weighted", zero_division=0),
    }


def run_scenario(source, scenario, classes):
    target_column = f"target_{scenario}"
    partitions, ranges = chronological_partitions(source)

    sequence_scaler = StandardScaler().fit(partitions["train"][SEQ_COLUMNS])
    context_preprocessor = ColumnTransformer([
        ("numeric", StandardScaler(), NUMERIC_CONTEXT),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_CONTEXT),
    ]).fit(partitions["train"][CONTEXT_COLUMNS])

    prepared = {}
    for name, part in partitions.items():
        transformed = part.copy()
        transformed.loc[:, SEQ_COLUMNS] = sequence_scaler.transform(part[SEQ_COLUMNS])
        x_seq, x_ctx_raw, y = create_windows(transformed, target_column)
        x_ctx = dense_float32(context_preprocessor.transform(pd.DataFrame(x_ctx_raw, columns=CONTEXT_COLUMNS)))
        prepared[name] = (x_seq, x_ctx, y)

    x_train_seq, x_train_ctx, y_train = prepared["train"]
    x_val_seq, x_val_ctx, y_val = prepared["validation"]
    x_test_seq, x_test_ctx, y_test = prepared["test"]

    flat_train = np.hstack([x_train_seq.reshape(len(x_train_seq), -1), x_train_ctx])
    flat_test = np.hstack([x_test_seq.reshape(len(x_test_seq), -1), x_test_ctx])
    metrics = []

    baselines = {
        "logistic_regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED),
        "random_forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, n_jobs=-1),
    }
    for name, estimator in baselines.items():
        estimator.fit(flat_train, y_train)
        metrics.append(metric_row(scenario, name, y_test, estimator.predict(flat_test)))

    model = build_hybrid_model(x_train_ctx.shape[1], classes)
    history = model.fit(
        {"seq_input": x_train_seq, "ctx_input": x_train_ctx}, y_train,
        validation_data=({"seq_input": x_val_seq, "ctx_input": x_val_ctx}, y_val),
        epochs=50, batch_size=32,
        callbacks=[EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)],
        verbose=1,
    )
    probabilities = model.predict({"seq_input": x_test_seq, "ctx_input": x_test_ctx}, verbose=0)
    prediction = probabilities.argmax(axis=1)
    metrics.append(metric_row(scenario, "hybrid_lstm", y_test, prediction))

    pd.DataFrame(history.history).to_csv(f"outputs/history_hybrid_lstm_{scenario}.csv", index=False)
    pd.DataFrame(
        confusion_matrix(y_test, prediction, labels=list(range(classes))),
        index=[f"actual_{i}" for i in range(classes)],
        columns=[f"predicted_{i}" for i in range(classes)],
    ).to_csv(f"outputs/confusion_matrix_hybrid_lstm_{scenario}.csv")
    model.save(f"models/hybrid_lstm_{scenario}.keras")
    joblib.dump(sequence_scaler, f"artifacts/sequence_scaler_{scenario}.joblib")
    joblib.dump(context_preprocessor, f"artifacts/context_preprocessor_{scenario}.joblib")
    Path(f"artifacts/metadata_{scenario}.json").write_text(json.dumps({
        "scenario": scenario,
        "classes": classes,
        "sequence_length": SEQUENCE_LENGTH,
        "sequence_columns": SEQ_COLUMNS,
        "context_columns": CONTEXT_COLUMNS,
        "chronological_ranges": ranges,
        "window_counts": {name: len(values[2]) for name, values in prepared.items()},
        "seed": SEED,
    }, indent=2), encoding="utf-8")
    return metrics


## 6. Run five-class and three-class scenarios

The three-class scenario maps original labels `0–1` to low (`0`), `2` to moderate (`1`), and `3–4` to high (`2`). The mapping is explicit and retained in the notebook for traceability.


In [ ]:
experiment_df = df.copy()
experiment_df["target_5_classes"] = experiment_df["congestion"].astype(int)
experiment_df["target_3_classes"] = experiment_df["congestion"].map({0: 0, 1: 0, 2: 1, 3: 2, 4: 2}).astype(int)

all_metrics = []
all_metrics.extend(run_scenario(experiment_df, "5_classes", 5))
all_metrics.extend(run_scenario(experiment_df, "3_classes", 3))
comparison = pd.DataFrame(all_metrics).sort_values(["scenario", "macro_f1"], ascending=[True, False])
comparison.to_csv("outputs/model_comparison_results.csv", index=False)
display(comparison)


## 7. Export traceable artifact package

This cell packages only files generated by the current run. Download the ZIP from the Colab file browser and retain it with the runtime summary for traceability.


In [ ]:
archive_base = "artifacts/boyaca_guayaquil_hybrid_lstm_artifacts"
shutil.make_archive(archive_base, "zip", root_dir=".", base_dir="outputs")

# Append models and fitted preprocessors to the same archive.
import zipfile
with zipfile.ZipFile(f"{archive_base}.zip", "a", compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in (Path("models"), Path("artifacts")):
        for path in folder.glob("*"):
            if path.is_file() and path.name != Path(f"{archive_base}.zip").name:
                archive.write(path, path.as_posix())

print("Artifact package:", f"{archive_base}.zip")
print("No metrics in this repository are precomputed; the displayed values come from this runtime.")
